In [1]:
import json
import math
import os
from openai import OpenAI
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from thefuzz import fuzz

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [3]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [4]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

OPENROUTER_MODEL = "openai/gpt-5.4"

In [5]:
def is_similar(text1, text2, threshold=80):
    return fuzz.partial_ratio(text1.strip(), text2.strip()) >= threshold

In [6]:
with open("final_golden_dataset.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [7]:
reranker_model = CrossEncoder('BAAI/bge-reranker-v2-m3')

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1221.71it/s]


In [8]:
def get_reranked_results(query, initial_results, k=5):
    points = initial_results.points
    pairs = [[query, res.payload['text']] for res in points]
    scores = reranker_model.predict(pairs)
    scored_results = sorted(zip(points, scores), key=lambda x: x[1], reverse=True)

    return [res[0] for res in scored_results[:k]]

In [9]:
def evaluate_retrieval_metrics(search_results, golden_context, k=5):
    hit = 0
    reciprocal_rank = 0
    relevant_found_count = 0
    dcg = 0
    
    for rank, result in enumerate(search_results[:k], start=1):
        if any(is_similar(golden_text, result) for golden_text in golden_context):
            if hit == 0:
                hit = 1
                reciprocal_rank = 1 / rank
            relevant_found_count += 1
            dcg += 1 / math.log2(rank + 1)
    recall = relevant_found_count / len(golden_context) if len(golden_context) > 0 else 0
    idcg = sum(1 / math.log2(i + 1) for i in range(1, min(len(golden_context), k) + 1))
    ndcg = dcg / idcg if idcg > 0 else 0
    return hit, reciprocal_rank, recall, ndcg

<hr>

## Evaluate baseline model

In [14]:
def get_metrics(model, collection, e5=False):
    total_hit = 0
    total_mrr = 0
    total_recall = 0
    total_ndcg = 0

    total_hit_reranked = 0
    total_mrr_reranked = 0
    total_recall_reranked = 0
    total_ndcg_reranked = 0

    for item in golden_data:
        if e5:
            query = model.encode("query: " + item['input']).tolist()
        else:
            query=model.encode(item['input']).tolist()
        results = client.query_points(
            collection_name=collection,
            prefetch=[
                models.Prefetch(
                    query=query,
                    using="default",
                    limit=20
                )
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=20
        )
        found_texts = [res.payload['text'] for res in results.points]
        
        h, m, r, n = evaluate_retrieval_metrics(found_texts, item['retrieval_context'])
        total_hit += h
        total_mrr += m
        total_recall += r
        total_ndcg += n

        reranked_objects = get_reranked_results(item['input'], results)
        reranked_texts = [res.payload['text'] for res in reranked_objects]

        h_rerank, m_rerank, r_rerank, n_rerank = evaluate_retrieval_metrics(reranked_texts, item['retrieval_context'])
        total_hit_reranked += h_rerank
        total_mrr_reranked += m_rerank
        total_recall_reranked += r_rerank
        total_ndcg_reranked += n_rerank

    return total_hit, total_mrr, total_recall, total_ndcg, \
            total_hit_reranked, total_mrr_reranked, total_recall_reranked, total_ndcg_reranked

In [26]:
MODEL_BASELINE = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
COLLECTION_BASELINE = "ucu_documents_baseline"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3228.37it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
total_hit_baseline, total_mrr_baseline, total_recall_baseline, total_ndcg_baseline, \
total_hit_reranked_baseline, total_mrr_reranked_baseline, total_recall_reranked_baseline, total_ndcg_reranked_baseline = get_metrics(MODEL_BASELINE, COLLECTION_BASELINE)

In [11]:
n = len(golden_data)

In [ ]:
print("Results for baseline model:")
print(f"Hit Rate @ 5: {round(total_hit_baseline/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline/n, 2)}")

Results for baseline model:
Hit Rate @ 5: 0.5
MRR @ 5: 0.31
Recall @ 5: 0.42
NDCG @ 5: 0.32


<hr>

## Evaluate baseline model with clever chunking

In [44]:
COLLECTION_BASELINE_RECURSIVE_CHAR = "ucu_documents_baseline_recursive_char"

In [31]:
total_hit_baseline_2, total_mrr_baseline_2, total_recall_baseline_2, total_ndcg_baseline_2, \
total_hit_reranked_baseline_2, total_mrr_reranked_baseline_2, total_recall_reranked_baseline_2, total_ndcg_reranked_baseline_2 = get_metrics(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR)

In [32]:
print("Results for baseline model (different chunking):")
print(f"Hit Rate @ 5: {round(total_hit_baseline_2/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline_2/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline_2/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline_2/n, 2)}")

Results for baseline model (different chunking):
Hit Rate @ 5: 0.45
MRR @ 5: 0.3
Recall @ 5: 0.47
NDCG @ 5: 0.36


Let`s see if the model performs better with reranker

In [33]:
print("Results for baseline model (different chunking + reranker):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_2/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_2/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_2/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_2/n, 2)}")

Results for baseline model (different chunking + reranker):
Hit Rate @ 5: 0.7
MRR @ 5: 0.64
Recall @ 5: 0.75
NDCG @ 5: 0.68


<hr>

## Evaluate baseline model with addition of sparse vectors - Hybrid Search

In [21]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")
COLLECTION_BASELINE_HYBRID = "ucu_documents_baseline_hybrid"

In [19]:
def get_metrics_sparse(model, collection, sparse_model, e5=False):
    total_hit = 0
    total_mrr = 0
    total_recall = 0
    total_ndcg = 0

    total_hit_reranked = 0
    total_mrr_reranked = 0
    total_recall_reranked = 0
    total_ndcg_reranked = 0

    for item in golden_data:
        if e5:
            dense_vector = model.encode("query: " + item['input']).tolist()
        else:
            dense_vector = model.encode(item['input']).tolist()
        sparse_emb = list(sparse_model.embed([item['input']]))[0]
            
        results = client.query_points(
            collection_name=collection,
            prefetch=[
                models.Prefetch(
                    query=dense_vector,
                    using="default",
                    limit=20
                ),
                models.Prefetch(
                    query=models.SparseVector(
                    indices=sparse_emb.indices.tolist(),
                    values=sparse_emb.values.tolist()
                ),
                    using="text_sparse",
                    limit=20
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=20
        )
        found_texts = [res.payload['text'] for res in results.points]
        
        h, m, r, n = evaluate_retrieval_metrics(found_texts, item['retrieval_context'])
        total_hit += h
        total_mrr += m
        total_recall += r
        total_ndcg += n

        reranked_objects = get_reranked_results(item['input'], results)
        reranked_texts = [res.payload['text'] for res in reranked_objects]

        h_rerank, m_rerank, r_rerank, n_rerank = evaluate_retrieval_metrics(reranked_texts, item['retrieval_context'])
        total_hit_reranked += h_rerank
        total_mrr_reranked += m_rerank
        total_recall_reranked += r_rerank
        total_ndcg_reranked += n_rerank

    return total_hit, total_mrr, total_recall, total_ndcg, \
            total_hit_reranked, total_mrr_reranked, total_recall_reranked, total_ndcg_reranked

In [15]:
total_hit_baseline_sparse, total_mrr_baseline_sparse, total_recall_baseline_sparse, total_ndcg_baseline_sparse, \
total_hit_reranked_baseline_sparse, total_mrr_reranked_baseline_sparse, total_recall_reranked_baseline_sparse, total_ndcg_reranked_baseline_sparse = get_metrics_sparse(MODEL_BASELINE, COLLECTION_BASELINE_HYBRID, MODEL_SPARSE)

In [16]:
print("Results for baseline model + sparse vectors added:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_sparse/n, 2)}")

Results for baseline model + sparse vectors added:
Hit Rate @ 5: 0.7
MRR @ 5: 0.66
Recall @ 5: 0.79
NDCG @ 5: 0.72


<hr>

## Evaluate baseline model with HyDE Search

In [29]:
def generate_hypothetical_answer(query, gemini_model):
    prompt = f"""
    Ти — асистент адміністрації Українського Католицького Університету (УКУ).

    Напиши логічну та лаконічну відповідь на запитання користувача.
    Використовуй офіційний стиль і термінологію, притаманну наказам та положенням УКУ.

    Сформулюй відповідь так, щоб вона могла містити:
    - ключові терміни та формулювання, які використовуються в офіційних документах
    - альтернативні або близькі за змістом формулювання (без значного збільшення обсягу)

    Не додавай зайвої інформації, але можеш узагальнювати формулювання для кращого покриття теми.

    Запитання: {query}
    """
    response = llm_client.chat.completions.create(
        model=gemini_model,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": query}
            ],
            temperature=0.1
    )
    return response.choices[0].message.content

In [31]:
def get_metrics_hyde(model, collection, openrouter_model, e5=False):
    total_hit = 0
    total_mrr = 0
    total_recall = 0
    total_ndcg = 0

    total_hit_reranked = 0
    total_mrr_reranked = 0
    total_recall_reranked = 0
    total_ndcg_reranked = 0

    for item in golden_data:
        hyde_doc = generate_hypothetical_answer(item['input'], openrouter_model)
        if e5:
            query_vector = model.encode("query: " + hyde_doc).tolist()
        else:
            query_vector = model.encode(hyde_doc).tolist()
            
        results = client.query_points(
            collection_name=collection,
            prefetch=[
                models.Prefetch(
                    query=query_vector,
                    using="default",
                    limit=20
                )
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=20
        )
        found_texts = [res.payload['text'] for res in results.points]
        
        h, m, r, n = evaluate_retrieval_metrics(found_texts, item['retrieval_context'])
        total_hit += h
        total_mrr += m
        total_recall += r
        total_ndcg += n

        reranked_objects = get_reranked_results(item['input'], results)
        reranked_texts = [res.payload['text'] for res in reranked_objects]

        h_rerank, m_rerank, r_rerank, n_rerank = evaluate_retrieval_metrics(reranked_texts, item['retrieval_context'])
        total_hit_reranked += h_rerank
        total_mrr_reranked += m_rerank
        total_recall_reranked += r_rerank
        total_ndcg_reranked += n_rerank

    return total_hit, total_mrr, total_recall, total_ndcg, \
            total_hit_reranked, total_mrr_reranked, total_recall_reranked, total_ndcg_reranked

In [87]:
total_hit_baseline_hyde, total_mrr_baseline_hyde, total_recall_baseline_hyde, total_ndcg_baseline_hyde, \
total_hit_reranked_baseline_hyde, total_mrr_reranked_baseline_hyde, total_recall_reranked_baseline_hyde, total_ndcg_reranked_baseline_hyde = get_metrics_hyde(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR, OPENROUTER_MODEL)

In [88]:
print("Results for baseline model + HyDe Search:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_hyde/n, 2)}")

Results for baseline model + HyDe Search:
Hit Rate @ 5: 0.65
MRR @ 5: 0.59
Recall @ 5: 0.62
NDCG @ 5: 0.58


<hr>

## Evaluate E5 small model (Naive)

In [12]:
MODEL_E5_SMALL = SentenceTransformer('intfloat/multilingual-e5-small')
COLLECTION_E5_SMALL = "ucu_documents_e5_small"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2249.91it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
total_hit_e5_small, total_mrr_e5_small, total_recall_e5_small, total_ndcg_e5_small, \
total_hit_reranked_e5_small, total_mrr_reranked_e5_small, total_recall_reranked_e5_small, total_ndcg_reranked_e5_small = get_metrics(MODEL_E5_SMALL, COLLECTION_E5_SMALL, e5=True)

In [25]:
print("Results for E5 small model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_small/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_small/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_small/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_small/n, 2)}")

Results for E5 small model:
Hit Rate @ 5: 0.8
MRR @ 5: 0.76
Recall @ 5: 0.9
NDCG @ 5: 0.81


<hr>

## Evaluate E5 small model (Hybrid)

In [17]:
COLLECTION_E5_SMALL_HYBRID = "ucu_documents_e5_small_hybrid"

In [26]:
total_hit_e5_small_sparse, total_mrr_e5_small_sparse, total_recall_e5_small_sparse, total_ndcg_e5_small_sparse, \
total_hit_reranked_e5_small_sparse, total_mrr_reranked_e5_small_sparse, total_recall_reranked_e5_small_sparse, total_ndcg_reranked_e5_small_sparse = get_metrics_sparse(MODEL_E5_SMALL, COLLECTION_E5_SMALL_HYBRID, sparse_model=MODEL_SPARSE, e5=True)

In [27]:
print("Results for E5 small model + sparse vectors:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_small_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_small_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_small_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_small_sparse/n, 2)}")

Results for E5 small model + sparse vectors:
Hit Rate @ 5: 0.8
MRR @ 5: 0.76
Recall @ 5: 0.92
NDCG @ 5: 0.82


<hr>

## Evaluate E5 small model (HyDE)

In [32]:
total_hit_e5_small_hyde, total_mrr_e5_small_hyde, total_recall_e5_small_hyde, total_ndcg_e5_small_hyde, \
total_hit_reranked_e5_small_hyde, total_mrr_reranked_e5_small_hyde, total_recall_reranked_e5_small_hyde, total_ndcg_reranked_e5_small_hyde = get_metrics_hyde(MODEL_E5_SMALL, COLLECTION_E5_SMALL, OPENROUTER_MODEL)

In [33]:
print("Results for E5 small model + HyDe Search:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_small_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_small_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_small_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_small_hyde/n, 2)}")

Results for E5 small model + HyDe Search:
Hit Rate @ 5: 0.75
MRR @ 5: 0.68
Recall @ 5: 0.88
NDCG @ 5: 0.76
